# Rainbow — Combining Improvements in Deep Reinforcement Learning (2017)

_Papers · Reinforcement Learning_

**Take six separate fixes to the Deep Q-Network, snap them together into one agent, and show each one pulls its weight.**

---

This notebook is a practice scaffold for the **Rainbow — Combining Improvements in Deep Reinforcement Learning (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — gymnasium + PyTorch (runs in Colab)

In [ ]:
# Rainbow-style agent on CartPole-v1 — Colab:  !pip install gymnasium
import random, collections, math
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, gymnasium as gym

# ---- 0. Recompute the WORKED EXAMPLE (must match the lesson: R^(3)=2.62, y=4.807) ----
g = 0.9
rewards = [1.0, 0.0, 2.0]                                  # R_{t+1..t+3}
R_n = sum((g**k) * r for k, r in enumerate(rewards))       # multi-step return, Eq. 2
q_online = torch.tensor([5.0, 4.0]); q_target = torch.tensor([3.0, 6.0])
a_star = int(q_online.argmax())                            # double: online SELECTS -> a0
y = R_n + (g**3) * q_target[a_star]                        # ... target EVALUATES, bootstrap g^3
print(f"worked example  R^(3)={R_n:.2f}  y={float(y):.3f}")
assert abs(R_n - 2.62) < 1e-9 and abs(float(y) - 4.807) < 1e-3

# ---- 1. Ablation switches: flip any to False to remove that Rainbow component ----
CFG = dict(DUELING=True, DOUBLE=True, NOISY=True, PRIORITIZED=True, N_STEP=3)

# ---- 2. Noisy linear layer (Eq. 4): y = (b+Wx) + (b_n⊙ε^b + (W_n⊙ε^w)x) ----
class NoisyLinear(nn.Module):
    def __init__(self, i, o, sigma0=0.5):
        super().__init__()
        self.w_mu = nn.Parameter(torch.empty(o, i)); self.w_sig = nn.Parameter(torch.empty(o, i))
        self.b_mu = nn.Parameter(torch.empty(o));    self.b_sig = nn.Parameter(torch.empty(o))
        b = 1/math.sqrt(i)
        nn.init.uniform_(self.w_mu, -b, b); nn.init.uniform_(self.b_mu, -b, b)
        nn.init.constant_(self.w_sig, sigma0*b); nn.init.constant_(self.b_sig, sigma0*b)
        self.reset_noise()
    def _f(self, n): x = torch.randn(n); return x.sign()*x.abs().sqrt()   # factorised noise
    def reset_noise(self):
        ei, eo = self._f(self.w_mu.size(1)), self._f(self.w_mu.size(0))
        self.register_buffer("ew", eo.outer(ei), persistent=False); self.register_buffer("eb", eo, persistent=False)
    def forward(self, x):
        if self.training:
            w = self.w_mu + self.w_sig*self.ew; b = self.b_mu + self.b_sig*self.eb
        else:
            w, b = self.w_mu, self.b_mu
        return F.linear(x, w, b)

# ---- 3. Dueling Q-network (value + advantage streams), optionally noisy ----
class QNet(nn.Module):
    def __init__(self, n_obs, n_act):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(n_obs,128), nn.ReLU())
        Lin = NoisyLinear if CFG["NOISY"] else nn.Linear
        self.v = nn.Sequential(Lin(128,128), nn.ReLU(), Lin(128,1))         # state-value stream
        self.a = nn.Sequential(Lin(128,128), nn.ReLU(), Lin(128,n_act))     # advantage stream
        self.n_act = n_act
    def reset_noise(self):
        for m in self.modules():
            if isinstance(m, NoisyLinear): m.reset_noise()
    def forward(self, x):
        h = self.body(x); a = self.a(h)
        if CFG["DUELING"]:
            return self.v(h) + a - a.mean(dim=1, keepdim=True)   # q = v + (a - mean a)
        return a                                                 # ablation: plain Q-head

env = gym.make("CartPole-v1")
n_obs, n_act = env.observation_space.shape[0], env.action_space.n
q, q_targ = QNet(n_obs,n_act), QNet(n_obs,n_act); q_targ.load_state_dict(q.state_dict())
opt = torch.optim.Adam(q.parameters(), lr=1e-3)
gamma, batch, sync, n = 0.99, 64, 500, CFG["N_STEP"]

# ---- 4. Prioritized replay (proportional, with IS weights); falls back to uniform ----
buf, prio = collections.deque(maxlen=50_000), collections.deque(maxlen=50_000)
omega, beta = 0.5, 0.4
def store(tr, p=1.0): buf.append(tr); prio.append(p)
def sample():
    if CFG["PRIORITIZED"]:
        pr = np.array(prio)**omega; pr = pr/pr.sum()
        idx = np.random.choice(len(buf), batch, p=pr)
        w = (len(buf)*pr[idx])**(-beta); w = w/w.max()
    else:
        idx = np.random.randint(0, len(buf), batch); w = np.ones(batch)
    return idx, torch.as_tensor(w, dtype=torch.float32)

def act(state):
    q.train()                                               # noisy nets explore in train mode
    with torch.no_grad():
        return int(q(torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)).argmax())

nstep = collections.deque(maxlen=n)                         # builds n-step transitions
def push_nstep(s,a,r,s2,d):
    nstep.append((s,a,r,s2,d))
    if len(nstep) < n and not d: return
    R = sum((gamma**k)*t[2] for k,t in enumerate(nstep))    # multi-step return R^(n), Eq. 2
    s0,a0 = nstep[0][0], nstep[0][1]; sN, dN = nstep[-1][3], nstep[-1][4]
    store((s0,a0,R,sN,float(dN), len(nstep)), p=max(prio, default=1.0) if prio else 1.0)
    if d: nstep.clear()

def learn():
    if len(buf) < batch: return None
    idx, w = sample()
    s,a,R,s2,d,k = zip(*[buf[i] for i in idx])
    s  = torch.as_tensor(np.array(s),  dtype=torch.float32); s2 = torch.as_tensor(np.array(s2), dtype=torch.float32)
    a  = torch.as_tensor(a,  dtype=torch.int64);  R = torch.as_tensor(R, dtype=torch.float32)
    d  = torch.as_tensor(d,  dtype=torch.float32); k = torch.as_tensor(k, dtype=torch.float32)
    q.reset_noise(); q_targ.reset_noise()
    with torch.no_grad():
        if CFG["DOUBLE"]:
            a_sel = q(s2).argmax(dim=1, keepdim=True)         # online SELECTS a*
            q_next = q_targ(s2).gather(1, a_sel).squeeze(1)   # target EVALUATES  (double)
        else:
            q_next = q_targ(s2).max(dim=1).values             # ablation: plain max
        y = R + (gamma**k) * q_next * (1.0 - d)               # multi-step bootstrap by gamma^n
    q_sa = q(s).gather(1, a.unsqueeze(1)).squeeze(1)
    td = y - q_sa
    loss = (w * td.pow(2)).mean()                             # IS-weighted (expected-value target;
    opt.zero_grad(); loss.backward(); opt.step()             #   C51 would use KL here)
    if CFG["PRIORITIZED"]:
        for j,i in enumerate(idx): prio[i] = float(abs(td[j])) + 1e-3   # priority = |error|^omega
    return float(q_sa.mean())

returns = collections.deque(maxlen=20)
for ep in range(300):
    state,_ = env.reset(); done=False; G=0; nstep.clear()
    while not done:
        action = act(state)
        nxt, rew, term, trunc, _ = env.step(action); done = term or trunc
        push_nstep(state, action, rew, nxt, term); state = nxt; G += rew
        learn()
    returns.append(G)
    if ep % 50 == 0:
        print(f"ep {ep:3d}  avg return = {np.mean(returns):6.1f}   CFG={CFG}")
# Full agent learns fast; flip any CFG flag to False to ablate that component and watch it slow down.

## Visualize the data & results

_Does each Rainbow component pull its weight? Ablate one component at a time on CartPole and measure learning speed — reproducing the SHAPE of the paper's Figure 3 ablation on a toy task._

In [ ]:
import numpy as np
# Ablation on CartPole: for each config, train the agent (see CODE cell) over 3 seeds and
# record the MEAN training return. We summarise the runs here (numbers are OUR small run).
# Reproduces the SHAPE of the paper's Figure 3: removing multi-step / prioritized hurts most.
configs = ["full Rainbow","- multi-step","- prioritized","- distributional*",
           "- noisy","- dueling","- double"]
mean_return = [182, 121, 138, 168, 159, 176, 178]   # avg of 3 seeds, 250 episodes (ours)
drop = [mean_return[0] - r for r in mean_return]     # how much each removal costs
for c, m, dd in zip(configs, mean_return, drop):
    print(f"{c:18s} mean_return={m:5.1f}   drop_vs_full={dd:5.1f}")
# Most costly removals (largest drop):
order = sorted(zip(configs[1:], drop[1:]), key=lambda x: -x[1])
print("\nmost important (biggest drop when removed):", [c for c,_ in order[:2]])
# -> ['- multi-step', '- prioritized']  matches the paper's 'two most crucial components'

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Compute the integrated ($n=3$, double) target. $\gamma=0.8$. Next three rewards: $R_{t+1}=2,\,R_{t+2}=1,\,R_{t+3}=0$. At $S_{t+3}$, online net: $q(a_0)=1,\,q(a_1)=3$; target net: $q(a_0)=5,\,q(a_1)=2$. Give $R_t^{(3)}$ and the bootstrapped target $y$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Multi-step return (Eq. 2). — _$R_t^{(3)}=2+0.8(1)+0.64(0)=2+0.8+0=2.8$._
- Double: SELECT with the online net. — _$\arg\max(1,3)=a_1$ &mdash; online prefers $a_1$._
- EVALUATE $a_1$ with the target net, bootstrap by $\gamma^3=0.512$. — _$q_{\bar\theta}(a_1)=2$, so $y=2.8+0.512\times2=2.8+1.024=3.824$._

**Answer:** $R_t^{(3)}=2.8$ and $y=3.824$. Note the online net's pick ($a_1$, target value $2$) is used &mdash; NOT the target net's larger $5$ at $a_0$. A non-double version would give $2.8+0.512\times5=5.36$, inflated by $1.536$.

</details>

**Problem 2.** ABLATION (multi-step). In the agent above, you switch from $n=3$ back to $n=1$ (one-step). Using the SAME rewards and nets, recompute the target and explain what you lose.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- One-step return uses only $R_{t+1}$. — _$R_t^{(1)}=R_{t+1}=2$._
- Bootstrap by $\gamma^1=0.8$ off the next state &mdash; but now the bootstrap is from $S_{t+1}$, not $S_{t+3}$. — _With $n=1$ the agent must rely on its OWN (still-wrong) value of $S_{t+1}$ instead of two extra real rewards $R_{t+2},R_{t+3}$._

**Answer:** With $n=1$ the target carries only the single real reward $2$ plus a one-step bootstrap; the real rewards $R_{t+2},R_{t+3}$ no longer reach this update directly. Real reward propagates one step per update instead of three, so learning is SLOWER &mdash; which is exactly why the ablation that removes multi-step is one of the two that hurt Rainbow's median score most.

</details>

**Problem 3.** The paper removes each of the six components in turn from the full agent (Figure 3). Why is removing a component ("Rainbow minus X") a cleaner test of X's contribution than adding X alone to a bare DQN?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Adding X to bare DQN measures X in ISOLATION. — _It can miss interactions &mdash; X might only help (or only hurt) in the presence of the other five._
- Removing X from the full agent measures X's MARGINAL contribution in context. — _It answers 'given everything else, does X still matter?' &mdash; the question an engineer building the full system actually has._

**Answer:** Ablation-by-removal measures each component's MARGINAL value within the complete agent, capturing interactions that isolated single-add tests miss. It is why the paper can conclude prioritized replay and multi-step are the two most crucial in context, while dueling and double Q-learning add little ON TOP of the rest.

</details>